In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mcstasscript.interface import instr,plotter,functions,reader
import mcstasscript as ms
import os

Cases:
- `MCPL_samples`: using `samples.mcpl.gz` without smearing
- `MCPL_resampled`: using `resampled.mcpl.gz` without smearing 
- `KDS_samples`: using `source.xml` without kde
- `KDS_resampled`: using `source.xml` with kde

Compilers:
- `clang`: without MPI
- `mpicc`: with MPI

`McStas` versions:
- Whatever is available in local conda env

In [ ]:
COMPILERS = ['clang', 'mpicc']
CASES = ['MCPL_samples', 'MCPL_resampled', 'KDS_samples', 'KDS_resampled']

In [ ]:
KDSPATH = os.getenv('CONDA_PREFIX')
os.environ['TMPDIR']='/tmp/'

In [ ]:
### Number of particles to be simulated (if MPI is used, it will be overwrite)
NPART = 100000000

In [ ]:
### Define the dimensions for the geometry
Rcol1 = 10 * 1e-2
Rcol2 = 10 * 1e-2

Lcol2 = Rcol2 * 5**0.5
Lcol1 = Lcol2 - 50 * 1e-2
if Lcol1 < 0:
    Lcol1 = 1e-2

Rmon = 20 * 1e-2

In [ ]:
### Remove older instruments
os.system('rm *.instr *.comp')

os.system('cp /Users/peterwillendrup/micromamba/envs/mcstas-dev/mcstas/contrib/KDSource.comp .')

### For each case, create an instrument with the same geometry, but changing the source definition
for CASE in CASES:
    
    Instr = instr.McStas_instr(name=CASE)
    Instr.author = 'Norberto Schmidt'
    Instr.origin = 'JCNS-HBS'

    origin = Instr.add_component('Origin', 'Arm', AT = [0, 0, 0])

    if CASE == 'MCPL_samples':
        source=Instr.add_component('Source','MCPL_input',AT=[0,0,0],RELATIVE=origin)
        source.filename="\"samples.mcpl.gz\""
        source.repeat_count=100
        source.verbose=0
        source.v_smear=0.02
        source.pos_smear=0.001
        source.dir_smear=0.01

    elif CASE == 'MCPL_resampled':
        source=Instr.add_component('Source','MCPL_input',AT=[0,0,0],RELATIVE=origin)
        source.filename="\"resampled.mcpl.gz\""
        source.repeat_count=100
        source.verbose=0
        source.v_smear=0.02
        source.pos_smear=0.001
        source.dir_smear=0.01

    elif CASE == 'KDS_samples':
        source=Instr.add_component('Source','KDSource',AT=[0,0,0],RELATIVE=origin)
        source.filename="\"source.xml\""
        source.use_kde=0
        source.repeat_count=100
        source.verbose=0

    elif CASE == 'KDS_resampled':
        source=Instr.add_component('Source','KDSource',AT=[0,0,0],RELATIVE=origin)
        source.filename="\"source.xml\""
        source.use_kde=1
        source.repeat_count=100
        source.verbose=0

    mon0 = Instr.add_component('Norm', 'Monitor', AT = [0, 0, 1e-2], RELATIVE = origin)
    mon0.xwidth = 2*Rmon
    mon0.yheight = 2*Rmon

    slit1 = Instr.add_component('Slit1', 'Slit', AT = [0, 0, Lcol1], RELATIVE = origin)
    slit1.radius = Rcol1

    # tracks1 = Instr.add_component('Tracks1', 'MCPL_output', AT=[0,0,0], RELATIVE = slit1)
    # tracks1.filename = "\"tracks1.mcpl\""
    # tracks1.doubleprec = 1
    # tracks1.verbose = 0

    lmon1 = Instr.add_component('LMon1', 'L_monitor', AT=[0,0,0], RELATIVE = slit1)
    lmon1.filename = "\"lmon1.dat\""
    lmon1.xwidth = 2*Rmon
    lmon1.yheight = 2*Rmon
    lmon1.Lmin = 0
    lmon1.Lmax = 5
    lmon1.nL = 100

    xymon1 = Instr.add_component('PSDMon1', 'PSD_monitor', AT=[0,0,0], RELATIVE = lmon1)
    xymon1.filename = "\"psdmon1.dat\""
    xymon1.xwidth = 2*Rmon
    xymon1.yheight = 2*Rmon
    xymon1.nx = 100
    xymon1.ny = 100

    slit2 = Instr.add_component('Slit2', 'Slit', AT = [0, 0, Lcol2], RELATIVE = origin)
    slit2.radius = Rcol2

    # tracks2 = Instr.add_component('Tracks2', 'MCPL_output', AT=[0,0,0], RELATIVE = slit2)
    # tracks2.filename = "\"tracks2.mcpl\""
    # tracks2.doubleprec = 1
    # tracks2.verbose = 0

    lmon2 = Instr.add_component('LMon2', 'L_monitor', AT=[0,0,0], RELATIVE = slit2)
    lmon2.filename = "\"lmon2.dat\""
    lmon2.xwidth = 2*Rmon
    lmon2.yheight = 2*Rmon
    lmon2.Lmin = 0
    lmon2.Lmax = 5
    lmon2.nL = 100

    xymon2 = Instr.add_component('PSDMon2', 'PSD_monitor', AT=[0,0,0], RELATIVE = lmon2)
    xymon2.filename = "\"psdmon2.dat\""
    xymon2.xwidth = 2*Rmon
    xymon2.yheight = 2*Rmon
    xymon2.nx = 100
    xymon2.ny = 100

    Instr.write_full_instrument()

    ### For both compilers, run the simulation
    for COMPILER in COMPILERS:
        FOLDER = CASE+'_'+COMPILER #+'_'+MCSTASVERSION.replace('.','')
        INSTR = Instr.name

        os.system('rm *.c *.out')
        os.system('rm -rf {:s}'.format(FOLDER))
        if COMPILER == 'clang':
            os.system('mcrun -c -y {:s} -d {:s} -n{:d}'.format(INSTR, FOLDER, NPART))
        elif COMPILER == 'mpicc':
            os.system('mcrun -c -y --mpi=auto {:s} -d {:s} -n{:d}'.format(INSTR, FOLDER, NPART))

    

In [ ]:
### Load the data with mcstasscript
DATA = {}
NORM = {}
for COMPILER in COMPILERS:
    for CASE in CASES:
        DATA['{:s}_{:s}'.format(CASE, COMPILER)] = functions.load_data('{:s}_{:s}'.format(CASE, COMPILER))
        NORM['{:s}_{:s}'.format(CASE, COMPILER)] = np.loadtxt('{:s}_{:s}/Norm.dat'.format(CASE, COMPILER))

### For all the results, normalize with the MCPL_samples_clang Monitor value
for KEY in NORM.keys():
    NORM[KEY] = np.append(NORM[KEY], NORM['MCPL_samples_clang'][0]/NORM[KEY][0])
    for data in DATA[KEY]:
        data.Intensity *= NORM[KEY][-1]
        data.Error *= NORM[KEY][-1]

In [ ]:
plot = plotter.make_sub_plot(DATA['KDS_resampled_clang'])

In [ ]:
plt.rcParams['figure.figsize']=(6.4*2, 4.8)
for key in DATA.keys():
    if key.find('clang')!=-1:
        plt.subplot(121)        
    else:
        plt.subplot(122)
    plt.errorbar(np.linspace(-Rmon*100, Rmon*100, 100), np.sum(DATA[key][2].Intensity, axis=0),np.sum(DATA[key][2].Error**2, axis=0)**0.5, label=key, ds='steps-mid')

plt.subplot(121)
plt.legend()
plt.grid()
plt.xlabel('x [cm]')
plt.ylabel('Intensity')

plt.subplot(122)
plt.legend()
plt.grid()
plt.xlabel('x [cm]')
plt.ylabel('Intensity')

plt.show()
plt.rcParams['figure.figsize']=(6.4, 4.8)

In [ ]:
plt.rcParams['figure.figsize']=(6.4*2, 4.8)
for key in DATA.keys():
    if key.find('clang')!=-1:
        plt.subplot(121)        
    else:
        plt.subplot(122)
    plt.errorbar(DATA[key][1].xaxis, DATA[key][1].Intensity, DATA[key][1].Error, ds='steps-mid', label=key)
plt.subplot(121)
plt.legend()
plt.grid()
plt.yscale('log')
plt.xlabel('Wavelength [AA]')
plt.ylabel('Intensity')

plt.subplot(122)
plt.legend()
plt.grid()
plt.yscale('log')
plt.xlabel('Wavelength [AA]')
plt.ylabel('Intensity')

plt.show()
plt.rcParams['figure.figsize']=(6.4, 4.8)